In [ ]:
# Cell 1 NumPy import and helper functions
import numpy as np


def _as_matrix(X):
    X = np.asarray(X, dtype=float)
    if X.ndim != 2 or X.shape[1] == 0 or not np.isfinite(X).all():
        raise ValueError("X must be a finite 2-D array with at least one feature.")
    return X


def _positive_integer(value, name):
    if isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, np.integer)) or value < 1:
        raise ValueError(f"{name} must be a positive integer.")
    return int(value)


def _fit_matrix(X, n_clusters):
    X = _as_matrix(X)
    if len(X) < n_clusters:
        raise ValueError("X must have at least n_clusters rows.")
    return X


def _prediction_matrix(X, centers):
    if centers is None:
        raise ValueError("Call fit before predict.")
    X = _as_matrix(X)
    if X.shape[1] != centers.shape[1]:
        raise ValueError("X must have the same number of features as the training data.")
    return X


def _squared_distances(X, centers):
    differences = X[:, None, :] - centers[None, :, :]
    return np.einsum("nkd,nkd->nk", differences, differences)

In [ ]:
# Cell 2 Standard K-Means
class CustomKMeans:
    """Lloyd's K-Means with uniform sampling of distinct row indices."""

    def __init__(self, n_clusters, *, tol=1e-4, max_iter=300, random_state=None):
        self.n_clusters = _positive_integer(n_clusters, "n_clusters")
        self.max_iter = _positive_integer(max_iter, "max_iter")
        if not np.isfinite(tol) or tol < 0:
            raise ValueError("tol must be finite and nonnegative.")
        self.tol = tol
        self.random_state = random_state

    def _initialize_centers(self, X, rng):
        indices = rng.choice(len(X), size=self.n_clusters, replace=False)
        return X[indices]

    def _run_lloyd(self, X, centers):
        for iteration in range(1, self.max_iter + 1):
            labels = _squared_distances(X, centers).argmin(axis=1)
            counts = np.bincount(labels, minlength=self.n_clusters)
            sums = np.zeros_like(centers)
            np.add.at(sums, labels, X)

            # An empty cluster contributes zero SSE. Preserve its center.
            occupied = counts > 0
            new_centers = centers.copy()
            new_centers[occupied] = sums[occupied] / counts[occupied, None]
            shift = np.linalg.norm(new_centers - centers, axis=1).max()
            centers = new_centers
            if shift <= self.tol:
                break

        # Reassign against the returned centers, including when max_iter is hit.
        distances = _squared_distances(X, centers)
        labels = distances.argmin(axis=1)
        inertia = float(distances[np.arange(len(X)), labels].sum())
        return centers, labels, inertia, iteration

    def fit(self, X):
        X = _fit_matrix(X, self.n_clusters)
        rng = np.random.default_rng(self.random_state)
        initial_centers = self._initialize_centers(X, rng)
        self.cluster_centers_, self.labels_, self.inertia_, self.n_iter_ = self._run_lloyd(
            X, initial_centers
        )
        return self

    def predict(self, X):
        centers = getattr(self, "cluster_centers_", None)
        X = _prediction_matrix(X, centers)
        return _squared_distances(X, centers).argmin(axis=1)

In [ ]:
# Cell 3 K-Means++
class CustomKMeansPlusPlus(CustomKMeans):
    """Use D² sampling to initialize the inherited Lloyd iteration."""

    def _initialize_centers(self, X, rng):
        indices = np.empty(self.n_clusters, dtype=int)
        chosen = np.zeros(len(X), dtype=bool)
        nearest_squared = np.full(len(X), np.inf)
        index = rng.integers(len(X))

        for position in range(self.n_clusters):
            indices[position] = index
            chosen[index] = True
            if position == self.n_clusters - 1:
                break

            squared = _squared_distances(X, X[index : index + 1])[:, 0]
            np.minimum(nearest_squared, squared, out=nearest_squared)
            nearest_squared[chosen] = 0.0
            weight = nearest_squared.sum()
            if weight > 0:
                index = rng.choice(len(X), p=nearest_squared / weight)
            else:
                # All data are covered by the chosen locations. Sample another row.
                index = rng.choice(np.flatnonzero(~chosen))

        return X[indices]